In [ ]:
import base64
import mimetypes
from pathlib import Path

from openai import OpenAI

In [ ]:
ollama_model="gemma4:latest"
#ollama_model="qwen3.5:9b"

# Configured by environment variables
ollama_base_url = "http://localhost:11434/v1"
ollama_api_key = "ollama"

client = OpenAI(api_key=ollama_api_key, base_url=ollama_base_url)

In [ ]:
system_prompt = """
Identify the single main object in the image: the foreground subject of the photograph.

Return only the name of the object, followed by concise descriptive details about its visible features, such as material, color, shape, texture, finish, or distinctive characteristics.

Do not mention background objects, secondary objects, people, scenery, or context. Do not provide explanations, confidence scores, introductions, labels, or any other extra text.

Output format:
[object name], [key visible features]
"""
user_prompt = "What is the main object in this image?"

In [ ]:
image_path = Path("../assets/PXL_20260813_144520511.jpg")

# Detect MIME type from the file extension
image_mime_type, _ = mimetypes.guess_type(image_path)

if image_mime_type is None:
    raise ValueError("Could not determine the image MIME type")

# Read the original file bytes — this preserves embedded metadata
with image_path.open("rb") as image_file:
    image_base64 = base64.b64encode(image_file.read()).decode("ascii")

data_uri = f"data:{image_mime_type};base64,{image_base64}"

In [ ]:
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": data_uri,
            },
            {"type": "text", "text": user_prompt},
        ],
    }
]

chat_response = client.chat.completions.create(
    model=ollama_model,
    messages=messages,
    max_tokens=256,
    temperature=1.0,
    top_p=0.95,
    
    extra_body={
        "top_k": 64,
        "reasoning":{"effort": "none"},
        "chat_template_kwargs": {"enable_thinking": False},
    },
)

print("Chat response:", chat_response)

In [ ]:
chat_response.choices[0].message.content

# NOW THE CORE of bordeus bot

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_ollama import ChatOllama

from langchain_postgres import PGVector
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
database_url = os.environ["DATABASE_URL"]

In [ ]:
def to_sqlalchemy_url(database_url: str) -> str:
    """`langchain-postgres` si aspetta un URL in stile SQLAlchemy
    (`postgresql+psycopg://...`), non lo schema `postgres://` usato nel
    resto del progetto (Go/pgx, `db.py`). Conversione automatica invece
    di dover mantenere due variabili d'ambiente diverse per la stessa
    connessione."""
    if database_url.startswith("postgresql+psycopg://"):
        return database_url
    for prefix in ("postgresql://", "postgres://"):
        if database_url.startswith(prefix):
            return "postgresql+psycopg://" + database_url[len(prefix) :]
    raise ValueError(f"DATABASE_URL non riconosciuto: {database_url!r}")

In [ ]:
comune_slug = "donnas"

model_name = "microsoft/harrier-oss-v1-0.6b"
embeddings = HuggingFaceEmbeddings(model_name=model_name)
vectorstore = PGVector(
        embeddings=embeddings,
        collection_name=comune_slug,
        connection=to_sqlalchemy_url(database_url),
        use_jsonb=True,
    )

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOllama(temperature=0.2, model=ollama_model)

In [ ]:
def identify_object(image_base64: str, image_mime_type: str):
    system_prompt = """
    Identify the single main object in the image: the foreground subject of the photograph.

    Return only the name of the object, followed by concise descriptive details about its visible features, such as material, color, shape, texture, finish, or distinctive characteristics.

    Do not mention background objects, secondary objects, people, scenery, or context. Do not provide explanations, confidence scores, introductions, labels, or any other extra text.

    Provide the information in italian language.

    Output format:
    [object name], [key visible features]
    """
    question = "Qual è l'oggetto principale in questa immagine?" # "What is the main object in this image?"
    system_message = SystemMessage(content=system_prompt)
    human_message = HumanMessage(
        content=[
            {"type": "text", "text": question},
            {
                "type": "image",
                "base64": image_base64,
                "mime_type": image_mime_type,
            },
        ]
    )

    messages = [system_message, human_message]
    print(messages)

    response = llm.invoke(messages)
    return response.content

In [ ]:
obj_descr = identify_object(image_base64, image_mime_type)

obj_descr

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing a waste management company.
You are chatting with a user about the the correct procedures for waste disposal.
Use only the context as knowledge. If relevant, use it to answer any question.
If you don't know the answer, say so.
Be concise. Don't mention the context. Just explain briefly the correct procedure for the waste disposal

Context:
{context}
"""

In [ ]:
retriever.invoke("tazza da caffè, ceramica bianca con decorazione nera stilizzata di un volto e la scritta ENBERG in nero")

In [ ]:
def answer_question(question: str):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    print(f"RAG Context: {context}")
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    system_message = SystemMessage(content=system_prompt)
    human_message = HumanMessage(
        content=[
            {"type": "text", "text": f"Come smaltisco questo oggetto: {question}?"}
        ]
    )

    messages = [system_message, human_message]
    print(messages)

    response = llm.invoke(messages)
    return response.content

In [ ]:
obj_descr = f"tazza da caffè, ceramica bianca con decorazione nera stilizzata di un volto e la scritta ENBERG in nero"

content = answer_question(obj_descr)

print(content)

In [ ]:
obj_descr = f"set di 18 piatti in ceramica bianca con decorazione nera stilizzata di un volto e la scritta ENBERG in nero"

content = answer_question(obj_descr)

print(content)